In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load processed dataset
df = pd.read_csv('../data/with_risk.csv')

# Features and target
X = df.drop(columns=['is_high_risk'])
y = df['is_high_risk']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [2]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def train_and_log_model(model, name, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        # Metrics
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds)
        rec = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, proba)

        # Log params and metrics
        mlflow.log_param("model_type", name)
        mlflow.log_metrics({
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "roc_auc": roc_auc
        })

        # Log model
        mlflow.sklearn.log_model(model, "model")
        print(f"✅ Logged: {name}, AUC: {roc_auc:.3f}")

# Example usage
train_and_log_model(LogisticRegression(max_iter=1000), "LogisticRegression", X_train, y_train, X_test, y_test)
train_and_log_model(RandomForestClassifier(n_estimators=100, random_state=42), "RandomForest", X_train, y_train, X_test, y_test)


c:\Users\user\Desktop\week5\Credit-Risk-Probability-Model\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2025/07/01 11:52:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/01 11:52:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Logged: LogisticRegression, AUC: 0.863


2025/07/01 11:52:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/01 11:53:02 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Logged: RandomForest, AUC: 0.999


In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='roc_auc')
grid.fit(X_train, y_train)

# Track best model
train_and_log_model(grid.best_estimator_, "RandomForest_GridSearch", X_train, y_train, X_test, y_test)


2025/07/01 11:55:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/07/01 11:55:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Logged: RandomForest_GridSearch, AUC: 0.999
